---
title: Automatic Piano Transcription Experiment
jupyter:
  jupytext:
    text_representation:
      extension: .qmd
      format_name: quarto
      format_version: '1.0'
      jupytext_version: 1.19.3
  kernelspec:
    display_name: Python 3
    language: python
    name: python3
---


This notebook is intentionally short. The implementation lives in editable Python modules under `src/piano_modeling/`, while realtime/deployment code lives under `src/piano_live_inference/`.


# Setup

In [1]:
#@title 1. Clone/update repo and install dependencies
import os

REPO_URL = "https://github.com/JDub323/piano_amt.git"  # change if this repo URL changes
REPO_DIR = "/content/piano_amt"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
%pip -q install -r requirements.txt
%pip -q install -e .

Cloning into '/content/piano_amt'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 124 (delta 45), reused 109 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 83.50 KiB | 5.22 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/piano_amt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 103.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 6.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for piano_amt (pyproject.toml) ... done


In [2]:
#@title 2. Autoreload project modules
import sys
import types
import importlib
imp = types.ModuleType("imp")
imp.reload = importlib.reload
sys.modules["imp"] = imp
%reload_ext autoreload
%autoreload 2

Remember: autoreload reloads module definitions, not objects. Recreate the `system`, loaders, optimizer, or config after changing their definitions.

In [3]:
#@title 3. Imports and runtime setup
from dataclasses import asdict
import gc
import json
from pathlib import Path

import pandas as pd
import torch

sys.path.insert(0, "/content/piano_amt/src")

from piano_modeling import Config, config_to_json, setup_runtime, make_colab_paths, print_paths
from piano_modeling.maestro_cache import load_cached_maestro_manifest, preprocess_maestro_in_download_batches
from piano_modeling.sliced_cache import load_or_build_sliced_dataset
from piano_modeling.models import PianoTranscriptionSystem, count_parameters_millions
from piano_modeling.training import run_training, load_checkpoint
from piano_modeling.benchmark import describe_runtime, sweep_training_settings, apply_best_benchmark_config
from piano_modeling.diagnostics import check_dataset_artifacts
from piano_modeling.export import export_prediction_bundle

DEVICE = setup_runtime(seed=1337)
describe_runtime(DEVICE)

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 95.0 GB
CUDA: 12.8
PyTorch: 2.11.0+cu128


In [4]:
#@title 4. Config
cfg = Config()

# Optional per-run overrides can still live here:
# cfg.resnet_name = "resnet34"
cfg.batch_size = 48
# cfg.num_workers = 2
# cfg.lr = 2e-4
# cfg.epochs = 40
cfg.compile_model = True

print(config_to_json(cfg))

{
  "maestro_version": "v3.0.0",
  "midi_min": 21,
  "midi_max": 108,
  "sample_rate": 16000,
  "segment_seconds": 12.0,
  "hop_length": 160,
  "n_fft": 2048,
  "n_mels": 229,
  "f_min": 27.5,
  "f_max": 8000.0,
  "feature_type": "mel",
  "cqt_bins_per_octave": 48,
  "cqt_n_bins": 384,
  "resnet_name": "resnet34.a1_in1k",
  "pretrained": true,
  "decoder_channels": 256,
  "decoder_type": "fpn",
  "dropout": 0.1,
  "compile_model": true,
  "FIND_OPTIMAL_SETTINGS": false,
  "batch_size": 48,
  "num_workers": 2,
  "epochs": 40,
  "lr": 0.0002,
  "weight_decay": 0.0001,
  "grad_clip": 1.0,
  "use_amp": true,
  "onset_loss_weight": 2.0,
  "offset_loss_weight": 1.0,
  "frame_loss_weight": 1.0,
  "velocity_loss_weight": 0.5,
  "sustain_loss_weight": 0.5,
  "onset_pos_weight": 16.0,
  "offset_pos_weight": 12.0,
  "frame_pos_weight": 3.0,
  "sustain_pos_weight": 2.0,
  "onset_width_frames": 2,
  "offset_width_frames": 2,
  "sustain_threshold": 64,
  "enable_audio_augment": false,
  "enable_spec

In [5]:
#@title 5. Mount Drive, create paths, and define cache-safety flags
from google.colab import drive

drive.mount('/content/drive')

paths = make_colab_paths('/content/drive/MyDrive/piano_transcription_resnet')
print_paths(paths)

# Safe defaults: use the existing pre-sliced dataset and do not redo expensive work.
USE_PRE_SLICED_DATASET = True
ALLOW_REBUILD_IF_SLICED_ZIP_MISSING = False
FORCE_RESPLICE_SLICED_CACHE = False

# Full-song MAESTRO preprocessing is fallback-only. Keep False unless intentionally rebuilding.
RUN_MAESTRO_PREPROCESS = False
NUM_MAESTRO_DOWNLOAD_BATCHES = 3
START_BATCH_INDEX = 0
END_BATCH_INDEX = -1
OVERWRITE_EXISTING_SPECS = False

Mounted at /content/drive
project_root: /content/drive/MyDrive/piano_transcription_resnet
datasets_root: /content/drive/MyDrive/piano_transcription_resnet/datasets
maestro_meta_root: /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_metadata
maestro_work_root: /content/maestro_working_audio
maestro_spec_root: /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_spectrogram_tensors
maestro_midi_root: /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_midi
maestro_cache_manifest: /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_spectrogram_tensors/maestro-v3.0.0_cached_manifest.csv
sliced_zip_path: /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_sliced.zip
sliced_root: /content/local_maestro_sliced
sliced_manifest: /content/local_maestro_sliced/sliced_manifest.csv
checkpoint_dir: /content/drive/MyDrive/piano_transcription_resnet/checkpoints
export_dir: /content/drive/MyD

In [6]:
#@title 6. Load metadata/full-song manifest without long downloads
# This downloads tiny MAESTRO metadata only if missing. It does not download the full audio dataset.
meta = load_cached_maestro_manifest(paths, cfg, download_if_missing=True)

if RUN_MAESTRO_PREPROCESS:
    meta = preprocess_maestro_in_download_batches(
        paths,
        cfg,
        num_batches=NUM_MAESTRO_DOWNLOAD_BATCHES,
        start_batch=START_BATCH_INDEX,
        end_batch=None if END_BATCH_INDEX < 0 else END_BATCH_INDEX,
        overwrite_existing_specs=OVERWRITE_EXISTING_SPECS,
        device=DEVICE,
    )
else:
    print('Full-song spectrogram preprocessing is disabled. The training path will restore/use the pre-sliced zip instead.')

metadata_csv: /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_metadata/maestro-v3.0.0.csv True
   canonical_composer                canonical_title       split  year  \
0          Alban Berg                   Sonata Op. 1       train  2018   
1          Alban Berg                   Sonata Op. 1       train  2008   
2          Alban Berg                   Sonata Op. 1       train  2017   
3  Alexander Scriabin  24 Preludes Op. 11, No. 13-24       train  2004   
4  Alexander Scriabin               3 Etudes, Op. 65  validation  2006   

                                       midi_filename  \
0  2018/MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R...   
1  2008/MIDI-Unprocessed_03_R2_2008_01-03_ORIG_MI...   
2  2017/MIDI-Unprocessed_066_PIANO066_MID--AUDIO-...   
3  2004/MIDI-Unprocessed_XP_21_R1_2004_01_ORIG_MI...   
4  2006/MIDI-Unprocessed_17_R1_2006_01-06_ORIG_MI...   

                                      audio_filename    duration  \
0  2018/MIDI-Unprocessed_Cham

In [7]:
#@title 7. Restore/load pre-sliced MAESTRO dataset
# Normal path: if /content/local_maestro_sliced exists, reuse it.
# Otherwise restore paths.sliced_zip_path from Drive.
# It only rebuilds/reslices if you explicitly enable the flags above.
sliced_meta = load_or_build_sliced_dataset(
    paths,
    cfg,
    meta_df=meta,
    use_pre_sliced_dataset=USE_PRE_SLICED_DATASET,
    allow_rebuild_if_sliced_zip_missing=ALLOW_REBUILD_IF_SLICED_ZIP_MISSING,
    force_resplice_sliced_cache=FORCE_RESPLICE_SLICED_CACHE,
)

Restoring pre-sliced dataset from Drive zip:
  /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_sliced.zip
-> /content/local_maestro_sliced
Restored sliced dataset to /content/local_maestro_sliced
Loading sliced manifest: /content/local_maestro_sliced/sliced_manifest.csv
Compatibility check passed by tensor shape for 9 sampled chunks. 9 sampled chunks were legacy files without metadata.
Sliced chunk counts:
split
train         47420
test           5935
validation     5782
Name: count, dtype: int64


In [8]:
#@title 8. Build model
system = PianoTranscriptionSystem(cfg).to(DEVICE)

if cfg.compile_model:
    system = torch.compile(system)

print('Parameters:', count_parameters_millions(system), 'M')

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Parameters: 27.802885 M


In [11]:
#@title 9. Optional benchmark batch size/workers
RUN_BENCHMARK = True  #@param {type:'boolean'}

torch.cuda.empty_cache()

if RUN_BENCHMARK:
    bench_df = sweep_training_settings(
        sliced_meta,
        base_cfg=cfg,
        batch_sizes=(16, 24, 32, 48, 64),
        num_workers_list=(2, 4, 8),
        amp_options=(True,),
        device=DEVICE,
    )
    display(bench_df)
    cfg = apply_best_benchmark_config(cfg, bench_df, memory_soft_limit_gb=14.0)
else:
    print('Skipping benchmark.')


Testing batch=16, workers=2, amp=True
Training chunks per epoch: 1024
Validation chunks: 1
{'batch_size': 16, 'num_workers': 2, 'use_amp': True, 'grad_accum_steps': 1, 'effective_batch_size': 16, 'lr': 0.0002, 'compile_model': True, 'status': 'ok', 'chunks_per_sec': 49.370495137580946, 'optimizer_steps_per_sec': 3.085655946098809, 'sec_per_optimizer_step': 0.3240802012500126, 'peak_gpu_gb': 18.9428391456604}

Testing batch=16, workers=4, amp=True
Training chunks per epoch: 1024
Validation chunks: 1
{'batch_size': 16, 'num_workers': 4, 'use_amp': True, 'grad_accum_steps': 1, 'effective_batch_size': 16, 'lr': 0.0002, 'compile_model': True, 'status': 'ok', 'chunks_per_sec': 49.402956566487546, 'optimizer_steps_per_sec': 3.0876847854054716, 'sec_per_optimizer_step': 0.3238672563749674, 'peak_gpu_gb': 18.9428391456604}

Testing batch=16, workers=8, amp=True
Training chunks per epoch: 1024
Validation chunks: 1
{'batch_size': 16, 'num_workers': 8, 'use_amp': True, 'grad_accum_steps': 1, 'eff

,batch_size,num_workers,use_amp,grad_accum_steps,effective_batch_size,lr,compile_model,status,chunks_per_sec,optimizer_steps_per_sec,sec_per_optimizer_step,peak_gpu_gb
0,48,2,True,1,48,0.0002,True,ok,50.015065,1.041981,0.959711,54.686389
1,48,8,True,1,48,0.0002,True,ok,49.969457,1.041030,0.960587,54.686389
2,48,4,True,1,48,0.0002,True,ok,49.749360,1.036445,0.964837,54.687403
3,32,2,True,1,32,0.0002,True,ok,49.711804,1.553494,0.643710,36.812510
4,32,4,True,1,32,0.0002,True,ok,49.643396,1.551356,0.644597,36.810770
5,32,8,True,1,32,0.0002,True,ok,49.624301,1.550759,0.644845,36.812510
6,24,4,True,1,24,0.0002,True,ok,49.495266,2.062303,0.484895,27.882288
7,24,2,True,1,24,0.0002,True,ok,49.446573,2.060274,0.485372,27.882288
8,16,4,True,1,16,0.0002,True,ok,49.402957,3.087685,0.323867,18.942839
9,64,8,True,1,64,0.0002,True,ok,49.400982,0.771890,1.295521,72.562460


Chosen config:
cfg.batch_size = 64
cfg.num_workers = 2
cfg.use_amp = True
batch_size                        64
num_workers                        2
use_amp                         True
grad_accum_steps                   1
effective_batch_size              64
lr                            0.0002
compile_model                   True
status                            ok
chunks_per_sec             49.332926
optimizer_steps_per_sec     0.770827
sec_per_optimizer_step      1.297308
peak_gpu_gb                72.562043
speed_ratio                 0.986361
Name: 12, dtype: object


In [17]:
#@title 10. Start or resume training
RUN_TRAINING = False    #@param {type:'boolean'}
CUSTOM_RUN_SUFFIX = "" #@param {type:'string'}
TRAIN_SAMPLES_PER_EPOCH = 20000 #@param {type:'integer'}
VAL_SAMPLES = 0 #@param {type:'integer'}
RESUME = True #@param {type:'boolean'}

torch.cuda.empty_cache()

if RUN_TRAINING:
    train_result = run_training(
        system=system,
        cfg=cfg,
        paths=paths,
        sliced_meta=sliced_meta,
        custom_run_suffix=CUSTOM_RUN_SUFFIX,
        train_samples_per_epoch=TRAIN_SAMPLES_PER_EPOCH,
        val_samples=VAL_SAMPLES,
        resume=RESUME,
        device=DEVICE,
        tensorboard_log_graph=False
    )
    RUN_NAME = train_result['run_name']
    best_ckpt = train_result['best_ckpt']
    last_ckpt = train_result['last_ckpt']
else:
    train_result = None
    RUN_NAME = None
    best_ckpt = None
    last_ckpt = None
    print('Skipping training.')

Training chunks per epoch: 20000
Validation chunks: 5782
Resumed from /content/drive/MyDrive/piano_transcription_resnet/checkpoints/resnet34.a1_in1k_maestro_v3.0.0_mel_sr16000_hop160_seg12/last.pt, starting epoch 5, best_metric=1.1133
Run directory: /content/drive/MyDrive/piano_transcription_resnet/checkpoints/resnet34.a1_in1k_maestro_v3.0.0_mel_sr16000_hop160_seg12
RUN_NAME: resnet34.a1_in1k_maestro_v3.0.0_mel_sr16000_hop160_seg12
TensorBoard log directory: /content/drive/MyDrive/piano_transcription_resnet/checkpoints/resnet34.a1_in1k_maestro_v3.0.0_mel_sr16000_hop160_seg12/tensorboard


train epoch 5:   0%|          | 0/312 [00:00<?, ?it/s]


[GPU Memory - Batch 0] Allocated: 1.23 GB, Reserved: 9.82 GB


validate:   0%|          | 0/91 [00:00<?, ?it/s]

Epoch 5: new best metric=1.1729; saved /content/drive/MyDrive/piano_transcription_resnet/checkpoints/resnet34.a1_in1k_maestro_v3.0.0_mel_sr16000_hop160_seg12/best.pt
{'epoch': 5, 'train_onset': 0.13494974004630095, 'train_offset': 0.08960320334881544, 'train_frame': 0.12016541943049584, 'train_sustain': 0.17336695784559616, 'train_velocity': 0.0036173838993701604, 'train_total': 0.5217027058586096, 'val_onset': 0.10342546258107554, 'val_offset': 0.07769979807393274, 'val_frame': 0.1072928917780132, 'val_sustain': 0.15961372804905624, 'val_velocity': 0.00271277486913477, 'val_total': 0.4507446521213417, 'val_onset_precision': 0.30336519479604934, 'val_onset_recall': 0.7408940614378852, 'val_onset_f1': 0.4304706325169297, 'val_offset_precision': 0.1110382281299983, 'val_offset_recall': 0.5369796729436488, 'val_offset_f1': 0.18402353183978587, 'val_frame_precision': 0.4751460915366106, 'val_frame_recall': 0.6771502166202773, 'val_frame_f1': 0.5584419155606417, 'val_sustain_precision': 0.8

...

validate:   0%|          | 0/91 [00:00<?, ?it/s]

{'epoch': 30, 'train_onset': 0.07656245931791954, 'train_offset': 0.06383379444909784, 'train_frame': 0.07343662911070845, 'train_sustain': 0.10810031660665305, 'train_velocity': 0.002560489203554029, 'train_total': 0.3244936875998974, 'val_onset': 0.06282297328026885, 'val_offset': 0.05945194980489758, 'val_frame': 0.07281548633723059, 'val_sustain': 0.13697781753721472, 'val_velocity': 0.00193955445715617, 'val_total': 0.33400778101827394, 'val_onset_precision': 0.34972101078344325, 'val_onset_recall': 0.8799918412803946, 'val_onset_f1': 0.5005260141784494, 'val_offset_precision': 0.11904440029421778, 'val_offset_recall': 0.7270942260752241, 'val_offset_f1': 0.204591761687809, 'val_frame_precision': 0.5062228413458756, 'val_frame_recall': 0.842645971758353, 'val_frame_f1': 0.6324805406249816, 'val_sustain_precision': 0.9058097506621425, 'val_sustain_recall': 0.9658252760470747, 'val_sustain_f1': 0.9348552896208601, 'selection_metric': 1.3375983164912402, 'best_metric': 1.369241994339

train epoch 31:   0%|          | 0/312 [00:00<?, ?it/s]


[GPU Memory - Batch 0] Allocated: 1.23 GB, Reserved: 87.24 GB


KeyboardInterrupt: 

# Inference/export examples

In [ ]:
#@title 11. Export a prediction bundle for an audio file
RUN_EXPORT_EXAMPLE = True #@param {type:'boolean'}
AUDIO_PATH = "/path/to/piano.wav" #@param {type:'string'}
OUT_STEM = "my_transcription" #@param {type:'string'}
CHECKPOINT_TO_LOAD = "best" #@param ["best", "last", "custom"]
CUSTOM_CHECKPOINT_PATH = "" #@param {type:'string'}

if RUN_EXPORT_EXAMPLE:
    if CHECKPOINT_TO_LOAD == 'custom':
        ckpt_path = Path(CUSTOM_CHECKPOINT_PATH)
    elif CHECKPOINT_TO_LOAD == 'last':
        ckpt_path = last_ckpt
    else:
        ckpt_path = best_ckpt
    load_checkpoint(ckpt_path, system, map_location=DEVICE, cfg=cfg)
    result = export_prediction_bundle(AUDIO_PATH, system, OUT_STEM, paths.export_dir, cfg)
    result
else:
    print('Skipping export example.')

# Diagnostics / optional live inference

In [ ]:
#@title 12. Check expected Drive/local artifacts
RUN_ARTIFACT_CHECK = False #@param {type:'boolean'}
if RUN_ARTIFACT_CHECK:
    check_dataset_artifacts(paths)
else:
    print('Skipping artifact check.')

In [ ]:
#@title 13. Optional realtime imports
# The realtime code is separated so you can ignore it until the model is good.
# from piano_live_inference import RealTimePianoTranscriber, benchmark_latency
# from piano_live_inference.gradio_app import launch_gradio_realtime
# from piano_live_inference.deploy import export_torchscript
#
# load_checkpoint(best_ckpt, system, map_location=DEVICE, cfg=cfg)
# benchmark_latency(system, cfg, seconds=2.0, repeats=20, device=DEVICE)
# launch_gradio_realtime(system, cfg)